# SnowHealth ML Pipeline — Pipeline Builder & Code Bundles

This notebook is a **lightweight companion** to the interactive notebook series (01–06). While those notebooks walk through each ML lifecycle step in detail, this notebook demonstrates how to **automate the entire workflow** as a scheduled, production-ready pipeline using three Snowflake features:

| Feature | What It Does |
|---|---|
| **Pipeline Builder** | Visual UI in Snowsight for designing task DAGs — drag-and-drop steps, set schedules, wire dependencies. Produces YAML that deploys via `snow dcm deploy`. |
| **Code Bundles** | Packaged Python scripts that run as Snowflake-managed ML Jobs. Each bundle has a `code_bundle.yml` (runtime config) and a `.py` entry point. The `ml` bundle type provides pandas, scikit-learn, and xgboost out of the box. |
| **DCM (Database Change Management)** | Infrastructure-as-code for Snowflake. A `manifest.yml` + `pipeline.yml` define all objects (tasks, code bundles, schedules) as version-controlled YAML — deployable via CLI or CI/CD with Jinja2 templating for multi-environment support. |

### Why This Matters for MLOps
In production ML, models need to be **retrained on a schedule**, **evaluated against quality gates**, and **promoted automatically** — not run manually from notebooks. Pipeline Builder + Code Bundles turn notebook logic into a repeatable, auditable, Git-versioned pipeline:

```
┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│  Feature Eng.   │───>│  Model Training │───>│  Model Deploy   │
│  (code bundle)  │    │  (code bundle)  │    │  (code bundle)  │
│  30+ features   │    │  XGBoost + reg  │    │  quality gate   │
│  from raw data  │    │  Model Registry │    │  promote default│
└─────────────────┘    └─────────────────┘    └─────────────────┘
         Weekly cron schedule (Monday 6 AM ET)
```

### Multi-Environment Templating
The pipeline definitions use `{{ variable }}` placeholders (Jinja2) that DCM resolves at deploy time from `manifest.yml`. This lets the **same YAML** deploy to DEV and PROD with different databases, warehouses, and compute pools — no code changes needed.

The `model_version` variable defaults to `"latest"` (auto-select newest from training) but can be overridden at deploy time to pin a specific version for production:
```bash
snow dcm deploy --from pipeline --target PROD --variable model_version=PIPELINE_20261012_v3
```

The `pipeline/` directory in this workspace contains the full DCM project. This notebook executes it and validates the results.

### Deploy the Pipeline (DCM)
The pipeline is defined as code in `pipeline/`:
- **`manifest.yml`** — targets (DEV/PROD), templating variables (`database`, `schema`, `warehouse`, `compute_pool`, `model_version`), asset mappings
- **`sources/definitions/pipeline.yml`** — templatized `resource: code_bundle` + `resource: task` definitions using `{{ variable }}` syntax
- **`code_bundles/*/code_bundle.yml`** — runtime config (`ml` type, `compute_pool`)
- **`code_bundles/*/*.py`** — self-contained ML logic (feature engineering, training, deployment)

Deployment is done via the **Snowflake CLI** (`snow dcm deploy`), which uploads the code, renders the Jinja2 templates, compiles the YAML, and creates/alters all Snowflake objects in one atomic operation. In a real MLOps workflow, this runs in CI/CD.

```bash
# DEV — auto-selects latest model version
snow dcm deploy --from pipeline --target DEV

# PROD — pin a tested version
snow dcm deploy --from pipeline --target PROD --variable model_version=PIPELINE_20261012_v3
```

In [ ]:
# Pipeline deployment is done via the snow CLI (outside the notebook runtime).
# See pipeline/README.md for deploy commands.
#
# If you need to deploy from here, run this in a Snowsight SQL worksheet:
#   !snow dcm deploy --from pipeline --target DEV
#
# The cells below execute and validate an already-deployed pipeline.
print('Pipeline assumed deployed — proceeding to execute and validate.')

### Resume Tasks and Execute Pipeline
Resumes all Snowflake Tasks in the DAG (leaf-first, root-last — required by Snowflake's task graph rules) and triggers an immediate run. In production, the **weekly cron schedule** (`0 6 * * 1 America/New_York`) handles this automatically.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

db_schema = 'SNOWHEALTH_ML_DB.SNOWHEALTH_MODELS'

# Resume tasks (leaf first, root last)
for task in ['SNOWHEALTH_TASK_MODEL_DEPLOYMENT', 'SNOWHEALTH_TASK_MODEL_TRAINING',
             'SNOWHEALTH_TASK_FEATURE_ENGINEERING', 'SNOWHEALTH_PIPELINE_ROOT']:
    session.sql(f'ALTER TASK {db_schema}.{task} RESUME').collect()
print('All tasks resumed')

# Execute
session.sql(f'EXECUTE TASK {db_schema}.SNOWHEALTH_PIPELINE_ROOT').collect()
print('Pipeline triggered — waiting for completion...')

### Wait for Pipeline Completion
Polls `TASK_HISTORY` every 30 seconds until all 4 tasks have completed. Each code bundle runs on `SNOWHEALTH_COMPUTE_POOL` (SPCS) and typically takes 1-2 minutes per step.

In [ ]:
import time

expected_tasks = {
    'SNOWHEALTH_PIPELINE_ROOT', 'SNOWHEALTH_TASK_FEATURE_ENGINEERING',
    'SNOWHEALTH_TASK_MODEL_TRAINING', 'SNOWHEALTH_TASK_MODEL_DEPLOYMENT'
}

for attempt in range(20):  # up to 10 minutes
    rows = session.sql(f"""
        SELECT NAME, STATE, 
               TIMESTAMPDIFF('second', SCHEDULED_TIME, COALESCE(COMPLETED_TIME, CURRENT_TIMESTAMP())) AS DURATION_S
        FROM TABLE(SNOWHEALTH_ML_DB.INFORMATION_SCHEMA.TASK_HISTORY(
            SCHEDULED_TIME_RANGE_START => DATEADD('minute', -15, CURRENT_TIMESTAMP())
        ))
        WHERE NAME IN ('SNOWHEALTH_PIPELINE_ROOT','SNOWHEALTH_TASK_FEATURE_ENGINEERING',
                       'SNOWHEALTH_TASK_MODEL_TRAINING','SNOWHEALTH_TASK_MODEL_DEPLOYMENT')
        AND SCHEDULED_TIME > DATEADD('minute', -10, CURRENT_TIMESTAMP())
        ORDER BY SCHEDULED_TIME ASC
    """).collect()

    completed = {r['NAME'] for r in rows if r['STATE'] in ('SUCCEEDED', 'FAILED')}
    executing = {r['NAME'] for r in rows if r['STATE'] == 'EXECUTING'}

    print(f"[{(attempt+1)*30}s] Completed: {len(completed)}/4  Executing: {list(executing)}")

    if completed >= expected_tasks:
        print('\nAll tasks finished!')
        break
    time.sleep(30)

# Print final results
print('\n' + '=' * 60)
for r in rows:
    status = '✓' if r['STATE'] == 'SUCCEEDED' else '✗'
    print(f"  {status} {r['NAME']:50s} {r['STATE']:10s} ({r['DURATION_S']}s)")
print('=' * 60)

### Validate Pipeline Outputs
Confirms the pipeline created the expected Snowflake objects — the same objects the interactive notebooks create, but produced automatically by the scheduled pipeline:
1. **Feature table** — `SNOWHEALTH_PIPELINE_FEATURES` (30+ engineered features)
2. **Model version** — new timestamped version in `SNOWHEALTH_PIPELINE_MODEL` (Model Registry)
3. **Deploy log** — entry with RMSE metric and DEPLOYED status (quality gate passed)

In [ ]:
print('=' * 60)
print('  PIPELINE VALIDATION')
print('=' * 60)

# 1. Feature table
try:
    feat_count = session.table('SNOWHEALTH_ML_DB.SNOWHEALTH_MODELS.SNOWHEALTH_PIPELINE_FEATURES').count()
    feat_cols = len(session.table('SNOWHEALTH_ML_DB.SNOWHEALTH_MODELS.SNOWHEALTH_PIPELINE_FEATURES').columns)
    print(f'  ✓ Feature table: {feat_count:,} rows, {feat_cols} columns')
except Exception as e:
    print(f'  ✗ Feature table: {e}')

# 2. Model registry
try:
    from snowflake.ml.registry import Registry
    reg = Registry(session=session, database_name='SNOWHEALTH_ML_DB', schema_name='SNOWHEALTH_MODELS')
    model = reg.get_model('SNOWHEALTH_PIPELINE_MODEL')
    versions = model.versions()
    latest = versions[-1]
    print(f'  ✓ Model: SNOWHEALTH_PIPELINE_MODEL ({len(versions)} version(s))')
    print(f'    Latest: {latest.version_name}')
    print(f'    Default: {model.default.version_name}')
except Exception as e:
    print(f'  ✗ Model: {e}')

# 3. Deploy log
try:
    log = session.sql("""
        SELECT VERSION, RMSE, STATUS, DEPLOYED_AT
        FROM SNOWHEALTH_ML_DB.SNOWHEALTH_MODELS.SNOWHEALTH_PIPELINE_DEPLOY_LOG
        ORDER BY DEPLOYED_AT DESC LIMIT 1
    """).collect()
    if log:
        r = log[0]
        print(f'  ✓ Deploy log: {r["VERSION"]} | RMSE={r["RMSE"]:.3f} | {r["STATUS"]}')
    else:
        print(f'  ✗ Deploy log: empty')
except Exception as e:
    print(f'  ✗ Deploy log: {e}')

print('=' * 60)

### Suspend Pipeline
Suspends the root task so the weekly cron schedule doesn't fire. In production, you'd leave this running. To trigger a manual run anytime: `EXECUTE TASK SNOWHEALTH_PIPELINE_ROOT`.

In [ ]:
session.sql(f'ALTER TASK {db_schema}.SNOWHEALTH_PIPELINE_ROOT SUSPEND').collect()
print('Pipeline suspended. To re-run manually:')
print(f'  EXECUTE TASK {db_schema}.SNOWHEALTH_PIPELINE_ROOT')